# 02_weather_silver

## Purpose

Transform Bronze weather observation records into a cleaned and reusable Silver table.

This notebook standardizes weather fields, casts numeric measurements, adds a reporting day, filters invalid records, and writes the result to the Silver layer.

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from transforms.weather_transforms import (
    standardize_weather_columns,
    cast_weather_fields,
    add_weather_day,
    filter_invalid_weather,
)

from validation.silver_checks import print_basic_quality_summary

In [0]:
catalog = "vattenfall_dev"

bronze_table = f"{catalog}.raw.bronze_weather"
silver_table = f"{catalog}.refined.silver_weather"

print("Source Bronze table:", bronze_table)
print("Target Silver table:", silver_table)

In [0]:
bronze_weather_df = spark.table(bronze_table)

before_count = bronze_weather_df.count()

print("Rows before cleaning:", before_count)
display(bronze_weather_df.limit(20))

In [0]:
silver_weather_df = (
    bronze_weather_df
    .transform(standardize_weather_columns)
    .transform(cast_weather_fields)
    .transform(add_weather_day)
    .transform(filter_invalid_weather)
)

after_count = silver_weather_df.count()

print("Rows after cleaning:", after_count)
print("Rows removed:", before_count - after_count)

display(silver_weather_df.limit(20))

In [0]:
(
    silver_weather_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("Written Silver table:", silver_table)

In [0]:
silver_df = spark.table(silver_table)

print_basic_quality_summary(silver_df, silver_table)

display(silver_df.limit(20))

In [0]:
print("Weather alert level values:")
silver_df.select("weather_alert_level").distinct().show(truncate=False)

print("Null check for key Silver fields:")
for column_name in ["region", "temperature_c", "wind_speed_kmh", "precipitation_mm", "weather_alert_level", "report_day"]:
    null_count = silver_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Invalid wind speed rows:")
invalid_wind_count = silver_df.filter("wind_speed_kmh < 0").count()
print("wind_speed_kmh < 0:", invalid_wind_count)